The onnx model is from: https://huggingface.co/justinchuby/Perch-onnx.
And the pytorch model is converted with the help of chatgpt

Application:  
  - finetune perchv2 model
  - enable use of audio file longer than 5 sec

Note: please study the pytorch code carefully. You may want to modify current tf compliant padding to pytorch friendly padding,

In [ ]:
try:
    import onnxruntime
except:
    !pip install onnxruntime


import sys
sys.path.append('/kaggle/input/datasets/hengck23/pytorch-perchv2')

from perchv2_onnx import *
from perchv2_pytorch import *
import librosa

KAGGLE_DIR = '/kaggle/input/competitions/birdclef-2026'


#load a demo audio file
audio, sr = librosa.load(
    f'{KAGGLE_DIR}/train_audio/greela/iNat1379393.ogg',
    sr=32000
)
t = len(audio) // 160_000
audio = audio[:t * 160_000].reshape(-1, 160_000)
audio = torch.from_numpy(audio).float()
print('audio:', audio.shape)

In [ ]:

#reference results from original onnx file
onnx_model = OnnxPerch2(
    '/kaggle/input/datasets/hengck23/pytorch-perchv2/perch_v2_no_dft.onnx'
)
spatial_embedding, embedding, spectrogram, label = onnx_model(audio)
print('onnx reference ---------------------------------')
print('embedding:', embedding.shape)
print('argmax:', embedding.argmax(-1))
print('max   :', embedding.max(-1))
print('values:', embedding.reshape(-1)[:10])


In [ ]:

#converted pytorch model
frontend = Perch2Frontend()
backbone = Perch2EfficientB3()

print(frontend.load_state_dict(
    torch.load('/kaggle/input/datasets/hengck23/pytorch-perchv2/perch_v2_spectrogram.pth', weights_only=True)))
print(backbone.load_state_dict(
    torch.load('/kaggle/input/datasets/hengck23/pytorch-perchv2/perch_v2_backbone.effb3.pth', weights_only=True)))

frontend.eval()
backbone.eval()

with torch.no_grad():
    spectrogram = frontend(audio)
    embedding = backbone(spectrogram)

print('pytorch ---------------------------------')
print('embedding:', embedding.shape)
print('argmax:', embedding.argmax(-1))
print('max   :', embedding.max(-1))
print('values:', embedding.reshape(-1)[:10])
